# BTD Makalesi — Doğrulama Notebook'u (Run All)

Bu notebook, makalede açık bırakılan **üç doğrulamayı** tek çalıştırmada tamamlar
ve sonuçları doğrudan LaTeX dosyalarına yapıştırılabilecek biçimde üretir.

| # | Doğrulama | Kapattığı itiraz | Yazılacağı yer |
|---|---|---|---|
| 1 | **S6 uygulama doğrulaması** | *"Sizin Mamba'nız Mamba değil"* | Bölüm 3.3, `\PH{CHECK 1 value}` ve `\PH{CHECK 2 value}` |
| 2 | **Bölme (split) doğrulaması** | *"Sızıntı garantisi iddia edilmiş, kanıtlanmamış"* | Bölüm 4.1, iki `\PH{...}` |
| 3 | **Sızıntı incelemesi** | *"NF-UNSW-NB15 sızıntılı görünüyor"* | Bölüm 5.7, bir `\PH{...}` |

**Tasarım ilkeleri** (ana notebook ile aynı):
- Her aşama kendi `try/except` bloğunda; biri başarısız olursa **diğerleri çalışmaya devam eder**.
- GPU **zorunlu değil**; hepsi CPU'da çalışır (S6 doğrulaması CPU'da birkaç saniye).
- `mamba-ssm` kurulamazsa **CHECK 1 atlanır, CHECK 2 yine de kanıt üretir**.
- Son hücre, `\PH{}` yuvalarına yapıştırılacak **LaTeX parçacıklarını** ve Drive'a bir rapor dosyası yazar.

**Çalıştırma:** `Runtime > Run all`. Tahmini süre: 5–15 dakika (çoğu 3. aşamanın RF eğitimi).

## Hücre 1 — Ortam ve durum kaydı

In [ ]:
# ============================================================
# HUCRE 1 — ORTAM
# GPU ZORUNLU DEGILDIR. Tum kontroller CPU'da calisir.
# ============================================================
import sys, platform, datetime, json, os, traceback

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
RESULTS = {}          # her asamanin sonucu buraya toplanir
STATUS  = {}          # asama -> "OK" / "SKIPPED: ..." / "FAILED: ..."

print(f"[{RUN_ID}] Dogrulama calismasi basladi (UTC).")
print(f"Python  : {platform.python_version()}")
print(f"Platform: {platform.platform()}")

try:
    import torch
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch : {torch.__version__} | cihaz: {DEV}")
    if DEV == "cuda":
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
except ImportError:
    torch = None
    DEV = "cpu"
    print("[UYARI] PyTorch bulunamadi. Asama 1 atlanacak, 2 ve 3 calisabilir.")

def stage(name):
    """Bir asamayi sarar: hata olursa yakalar, notebook AKISINI KESMEZ."""
    class _Ctx:
        def __enter__(self):
            print("\n" + "=" * 68)
            print(f"ASAMA: {name}")
            print("=" * 68)
            return self
        def __exit__(self, et, ev, tb):
            if et is None:
                STATUS.setdefault(name, "OK")
            else:
                STATUS[name] = f"FAILED: {et.__name__}: {ev}"
                print(f"\n[HATA] {name} basarisiz oldu -> {et.__name__}: {ev}")
                traceback.print_exc(limit=6)
                print("[BILGI] Diger asamalar calismaya devam ediyor.")
            return True     # istisnayi yut
    return _Ctx()

## Hücre 2 — Drive bağlama ve proje klasörünü **otomatik bulma**

Bu hücre sabit bir yola güvenmez. Drive'ı bağlamayı dener, sonra proje klasörünü
**birkaç aday adla** ve ayrıca **veri dosyalarının kendisini arayarak** bulur —
klasör adı farklıysa veya iç içe bir yapıdaysa da çalışır.

Bulamazsa ne gördüğünü listeler. Yolu biliyorsanız `MANUEL_BASE_DIR` alanına
yazmanız yeterlidir; arama tamamen atlanır.

In [ ]:
# ============================================================
# HUCRE 2 — DRIVE + PROJE KLASORUNU/VERIYI OTOMATIK BULMA
# ============================================================
import glob

MANUEL_BASE_DIR = ""      # <-- yolu BILIYORSANIZ buraya yazin, arama atlanir

# Proje klasoru icin aday adlar (buyuk/kucuk harf duyarsiz).
# Mamba_BTD ONCE gelir: ana notebook'un kullandigi klasor budur.
ADAY_ADLAR = ["Mamba_BTD", "BTD_Mamba_NIDS_Robustness", "BTD_Mamba", "Mamba_NIDS"]

# Veri dosyalarini TANIYAN desenler: klasor yapisi ne olursa olsun bunlar aranir
VERI_DESENLERI = {
    "NF-UNSW-NB15": ["NF-UNSW-NB15*.csv", "NF_UNSW_NB15*.csv"],
    "CICIDS2018":   ["CICIDS2018*.csv", "*CSE-CIC*2018*.csv", "*CIC*IDS*2018*.csv"],
    "NSL-KDD":      ["KDDTrain+*.txt", "KDDTrain*.txt"],
}

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("[OK] drive.mount calisti.")
except Exception as e:
    print(f"[BILGI] drive.mount calistirilmadi ({type(e).__name__}). "
          f"Colab disindaysaniz NORMALDIR; arama yine de yapilacak.")

KOKLER = [r for r in ["/content/drive/MyDrive", "/content/drive/My Drive",
                      "/content/drive/Shareddrives", "/content/drive",
                      "/content", os.getcwd()] if os.path.isdir(r)]
print(f"\n[TARAMA] taranacak kokler: {KOKLER}")

def _ada_gore():
    hedef = {a.lower() for a in ADAY_ADLAR}
    for r in KOKLER:
        for depth in ("/*", "/*/*", "/*/*/*"):
            for hit in glob.glob(r + depth):
                if os.path.isdir(hit) and os.path.basename(hit).lower() in hedef:
                    return hit
    return None

def _veriye_gore():
    """Imza dosyasini bul, ondan proje kokunu geriye dogru cikar."""
    for r in KOKLER:
        for pat in ["**/NF-UNSW-NB15*.csv", "**/KDDTrain+*.txt"]:
            for hit in glob.glob(os.path.join(r, pat), recursive=True):
                d = os.path.dirname(hit)
                # .../<BASE>/datasets/<NAME>/file  ya da  .../<BASE>/datasets/file
                for up in (3, 2):
                    cand = d
                    for _ in range(up - 1):
                        cand = os.path.dirname(cand)
                    if os.path.basename(os.path.dirname(hit)).lower() != "datasets":
                        cand = os.path.dirname(os.path.dirname(d))
                    if os.path.isdir(cand):
                        return cand
    return None

if MANUEL_BASE_DIR:
    BASE_DIR, nasil = MANUEL_BASE_DIR, "manuel"
else:
    BASE_DIR = _ada_gore()
    nasil = "klasor adi"
    if not BASE_DIR:
        BASE_DIR = _veriye_gore()
        nasil = "veri dosyasindan geriye"

DRIVE_OK = bool(BASE_DIR) and os.path.isdir(BASE_DIR)
if DRIVE_OK:
    print(f"\n[OK] Proje klasoru bulundu ({nasil}):\n     {BASE_DIR}")
else:
    print("\n[UYARI] Proje klasoru BULUNAMADI -> Asama 2 ve 3 atlanacak.")
    print("        Asagidaki listeden dogru yolu bulup MANUEL_BASE_DIR'e yazin.\n")
    for r in KOKLER:
        try:
            print(f"  {r}  ->  {sorted(os.listdir(r))[:30] or '(bos)'}")
        except Exception as ex:
            print(f"  {r}  ->  okunamadi ({type(ex).__name__})")

# ---- Veri dosyalarini BASE_DIR altinda ADIYLA ara (klasor yapisindan bagimsiz) ----
DATA, OUT_DIR = {}, None
if DRIVE_OK:
    OUT_DIR = f"{BASE_DIR}/verification"
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"\n[OK] Cikti klasoru: {OUT_DIR}\n")
    for ad, desenler in VERI_DESENLERI.items():
        hits = []
        for dsn in desenler:
            hits += glob.glob(os.path.join(BASE_DIR, "**", dsn), recursive=True)
        hits = sorted(set(hits))
        # '/datasets/' altindaki kopyayi TERCIH ET (ana notebook'un yapisi)
        tercih = [h for h in hits if os.sep + "datasets" + os.sep in h]
        bulunan = (tercih or hits)[0] if hits else None
        if bulunan:
            DATA[ad] = bulunan
            print(f"  {ad:14s} -> VAR  {bulunan}")
            if len(hits) > 1:
                print(f"       [DIKKAT] {len(hits)} kopya bulundu; digerleri: "
                      f"{[h for h in hits if h != bulunan][:3]}")
        else:
            print(f"  {ad:14s} -> YOK  (desenler: {desenler})")
    if not DATA:
        print("\n  [BILGI] BASE_DIR altindaki ilk seviye icerik:")
        try:
            print(f"          {sorted(os.listdir(BASE_DIR))[:30]}")
        except Exception:
            pass


## Hücre 3 — Resmî referans tarama fonksiyonunu temin etme

CHECK 1, resmî `mamba-ssm` deposundaki `selective_scan_ref` fonksiyonuyla
karşılaştırma yapar. **Önemli nokta:** bu fonksiyon saf PyTorch'tur; özel CUDA
çekirdeklerine ihtiyaç duymaz. Paketin tamamını kurmak ise CUDA derlemesi
gerektirir ve sıklıkla başarısız olur.

Bu hücre bu nedenle üç yolu sırayla dener:

1. Paket zaten kuruluysa doğrudan içe aktar.
2. Değilse, resmî depodan **yalnızca ilgili kaynak dosyayı indirip** içinden
   `selective_scan_ref` fonksiyonunu ayıkla (derleme gerekmez).
3. O da olmazsa `--no-build-isolation` ile paketi kurmayı dene.

Hiçbiri olmazsa CHECK 1 atlanır; CHECK 2 tek başına kanıt üretmeye devam eder.

In [ ]:
# ============================================================
# HUCRE 3 — RESMI REFERANS TARAMA FONKSIYONU
# selective_scan_ref SAF PyTorch'tur; CUDA derlemesi GEREKTIRMEZ.
# ============================================================
import ast, subprocess, urllib.request
import torch                      # Hucre 1'de de yuklendi; hucre bagimsiz calisabilsin

selective_scan_ref = None
KAYNAK = "kullanilamadi"
REF_URL = ("https://raw.githubusercontent.com/state-spaces/mamba/main/"
           "mamba_ssm/ops/selective_scan_interface.py")

# ---- Yol 1: paket zaten kurulu mu? ----
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_ref
    KAYNAK = "kurulu mamba-ssm paketi"
except Exception:
    pass

# ---- Yol 2: yalnizca kaynak dosyayi indir, fonksiyonu ayikla ----
if selective_scan_ref is None:
    try:
        print("[BILGI] Paket yok. Resmi kaynak dosya indiriliyor (derleme YOK)...")
        src = urllib.request.urlopen(REF_URL, timeout=60).read().decode("utf-8")
        tree = ast.parse(src)
        fn = next(n for n in tree.body
                  if isinstance(n, ast.FunctionDef) and n.name == "selective_scan_ref")
        from einops import rearrange, repeat
        ns = {"torch": torch, "F": torch.nn.functional,
              "rearrange": rearrange, "repeat": repeat}
        exec(ast.get_source_segment(src, fn), ns)
        selective_scan_ref = ns["selective_scan_ref"]
        KAYNAK = "resmi depodan ayiklanan kaynak (state-spaces/mamba)"
        print("[OK] selective_scan_ref resmi kaynaktan ayiklandi.")
    except Exception as e:
        print(f"[BILGI] Kaynak ayiklama basarisiz: {type(e).__name__}: {e}")

# ---- Yol 3: son care, --no-build-isolation ile kurulum ----
if selective_scan_ref is None:
    print("[BILGI] Son care: pip install --no-build-isolation mamba-ssm ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--no-cache-dir", "--no-build-isolation", "mamba-ssm"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        try:
            from mamba_ssm.ops.selective_scan_interface import selective_scan_ref
            KAYNAK = "mamba-ssm (--no-build-isolation)"
            print("[OK] Kurulum basarili.")
        except Exception as e:
            print(f"[BILGI] Kuruldu ama ice aktarilamadi: {e}")
    else:
        print("[BILGI] Kurulum basarisiz (CUDA derlemesi gerektirir). "
              "Bu BEKLENEN bir durumdur.")

MAMBA_SSM_OK = selective_scan_ref is not None
print(f"\nCHECK 1 (resmi referansla karsilastirma): "
      f"{'CALISACAK  [kaynak: ' + KAYNAK + ']' if MAMBA_SSM_OK else 'ATLANACAK'}")
print("CHECK 2 (kapali-form karsilastirma)      : HER HALUKARDA CALISACAK")


## Hücre 4 — Makaledeki S6 bloğu (Cell 7'den birebir)

Aşağıdaki sınıf ana notebook'un Hücre 7'sinden **değiştirilmeden** alınmıştır;
tek fark, ara operatörleri (`dA`, `dB`, `Cm`, `delta`) dışarı veren bir yardımcı
metodun eklenmiş olmasıdır — bu, ileri geçişin matematiğini değiştirmez, yalnızca
karşılaştırma için erişim sağlar.

In [ ]:
# ============================================================
# HUCRE 4 — MAKALEDEKI S6 BLOGU (Cell 7 ile ayni matematik)
# ============================================================
if torch is not None:
    import torch.nn as nn
    import torch.nn.functional as F

    class SelectiveSSMBlock(nn.Module):
        def __init__(self, d_model: int, d_state: int = 16, expand: int = 2):
            super().__init__()
            self.d_model = d_model
            self.d_inner = expand * d_model
            self.d_state = d_state
            self.in_proj  = nn.Linear(d_model, 2 * self.d_inner)
            self.conv1d   = nn.Conv1d(self.d_inner, self.d_inner,
                                      kernel_size=4, padding=3, groups=self.d_inner)
            self.x_proj   = nn.Linear(self.d_inner, self.d_state * 2 + self.d_inner)
            self.dt_proj  = nn.Linear(self.d_inner, self.d_inner)
            A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
            self.A_log    = nn.Parameter(torch.log(A))
            self.D        = nn.Parameter(torch.ones(self.d_inner))
            self.out_proj = nn.Linear(self.d_inner, d_model)
            self.norm     = nn.LayerNorm(d_model)

        def operators(self, x):
            """Taramaya kadar olan her sey. Karsilastirma icin disari verilir."""
            B, L, _ = x.shape
            xz = self.in_proj(x)
            x_in, z = xz.chunk(2, dim=-1)
            x_conv = self.conv1d(x_in.transpose(1, 2))[:, :, :L]
            x_conv = F.silu(x_conv.transpose(1, 2))
            dbc = self.x_proj(x_conv)
            delta, Bm, Cm = torch.split(
                dbc, [self.d_inner, self.d_state, self.d_state], dim=-1)
            delta = F.softplus(self.dt_proj(delta))
            A  = -torch.exp(self.A_log)
            dA = torch.exp(delta.unsqueeze(-1) * A)
            dB = delta.unsqueeze(-1) * Bm.unsqueeze(2)
            return dict(x_conv=x_conv, z=z, dA=dA, dB=dB, Bm=Bm, Cm=Cm,
                        delta=delta, A=A)

        def scan_loop(self, x_conv, dA, dB, Cm):
            """Makaledeki ardisik tarama (Algoritma 2, satir 7-10)."""
            B, L, _ = x_conv.shape
            h = torch.zeros(B, self.d_inner, self.d_state,
                            device=x_conv.device, dtype=x_conv.dtype)
            ys = []
            for t in range(L):
                h = dA[:, t] * h + dB[:, t] * x_conv[:, t].unsqueeze(-1)
                ys.append(torch.einsum("bds,bs->bd", h, Cm[:, t]))
            return torch.stack(ys, dim=1)

        def forward(self, x):
            residual = x
            o = self.operators(x)
            y = self.scan_loop(o["x_conv"], o["dA"], o["dB"], o["Cm"])
            y = y + o["x_conv"] * self.D
            y = y * F.silu(o["z"])
            return self.norm(self.out_proj(y) + residual)

    print("[OK] SelectiveSSMBlock tanimlandi.")
else:
    print("[ATLANDI] PyTorch yok.")

## Hücre 5 — AŞAMA 1: S6 uygulama doğrulaması

**CHECK 1** — Resmî `selective_scan_ref` ile karşılaştırma. Aynı girdiler, aynı
ayrıklaştırılmış operatörler; elemanlar arası maksimum mutlak sapma.

**CHECK 2** — Kapalı-form karşılaştırma. Diyagonal $A$ için özyinelemenin çözümü

$$h_t=\sum_{s\le t}\Bigl(\prod_{r=s+1}^{t}\bar A_r\Bigr)\odot\bar B_s\odot u_s$$

olup, burada $O(L^2)$ doğrudan toplamla, **hiç döngüsel durum taşımadan** hesaplanır.
Bu, sizin $O(L)$ döngünüzden tamamen bağımsız bir uygulamadır: ikisi aynı sayıyı
veriyorsa tarama matematiği doğrudur.

**CHECK 3** — Sonlu farklarla gradyan kontrolü. Bozuk bir geri geçiş her FGSM/PGD
sonucunu bozardı, dolayısıyla bu kontrol beyaz-kutu bulgularının önkoşuludur.

In [ ]:
# ============================================================
# HUCRE 5 — ASAMA 1: S6 DOGRULAMASI
# ============================================================
with stage("1 — S6 uygulama dogrulamasi"):
    assert torch is not None, "PyTorch yok"
    torch.manual_seed(0)
    d_model, d_state, Bsz, L = 64, 16, 4, 64

    # ---------- CHECK 1 ----------
    if MAMBA_SSM_OK:
        blk = SelectiveSSMBlock(d_model, d_state).to(DEV).eval()
        x = torch.randn(Bsz, L, d_model, device=DEV)
        with torch.no_grad():
            o = blk.operators(x)
            ours = blk.scan_loop(o["x_conv"], o["dA"], o["dB"], o["Cm"])
            # selective_scan_ref: u,delta -> (B,D,L); B,C -> (B,N,L); A -> (D,N)
            ref = selective_scan_ref(
                o["x_conv"].transpose(1, 2).contiguous(),
                o["delta"].transpose(1, 2).contiguous(),
                o["A"],
                o["Bm"].transpose(1, 2).contiguous(),
                o["Cm"].transpose(1, 2).contiguous(),
                D=None, z=None, delta_bias=None,
                delta_softplus=False, return_last_state=False,
            ).transpose(1, 2)
            dev1 = (ours - ref).abs().max().item()
        RESULTS["check1"] = dev1
        print(f"CHECK 1  {'GECTI' if dev1 < 1e-4 else 'KALDI'}  "
              f"max|bizim - selective_scan_ref| = {dev1:.3e}   (fp32 esik 1e-4)")
    else:
        RESULTS["check1"] = None
        print("CHECK 1  ATLANDI (mamba-ssm yok)")

    # ---------- CHECK 2 ----------
    blk64 = SelectiveSSMBlock(d_model, d_state).to(DEV).double().eval()
    x64 = torch.randn(Bsz, L, d_model, device=DEV, dtype=torch.float64)
    with torch.no_grad():
        o = blk64.operators(x64)
        dA, dB, u, Cm = o["dA"], o["dB"], o["x_conv"], o["Cm"]
        loop = blk64.scan_loop(u, dA, dB, Cm)

        # Bagimsiz O(L^2) kapali-form: durum tasinmaz, her t icin sifirdan toplanir
        src = dB * u.unsqueeze(-1)                     # (B,L,d_inner,N)
        ys = []
        for t in range(L):
            acc = torch.zeros_like(src[:, 0])
            for s in range(t + 1):
                w = src[:, s]
                if s < t:
                    w = w * dA[:, s + 1:t + 1].prod(dim=1)
                acc = acc + w
            ys.append(torch.einsum("bds,bs->bd", acc, Cm[:, t]))
        closed = torch.stack(ys, dim=1)
        dev2 = (loop - closed).abs().max().item()
    RESULTS["check2"] = dev2
    print(f"CHECK 2  {'GECTI' if dev2 < 1e-8 else 'KALDI'}  "
          f"max|dongu - kapali form| = {dev2:.3e}   (fp64 esik 1e-8)")

    # ---------- CHECK 3 ----------
    small = SelectiveSSMBlock(8, 4).to(DEV).double()
    xg = torch.randn(2, 6, 8, device=DEV, dtype=torch.float64, requires_grad=True)
    ok3 = torch.autograd.gradcheck(lambda t: small(t).sum(), (xg,),
                                   eps=1e-6, atol=1e-5, rtol=1e-3)
    RESULTS["check3"] = bool(ok3)
    print(f"CHECK 3  {'GECTI' if ok3 else 'KALDI'}  gradcheck (fp64, tam blok)")

## Hücre 6 — AŞAMA 2: Bölme (split) doğrulaması

Bölme, verilen tohumda **deterministiktir**, dolayısıyla ham korpuslardan yeniden
türetilip grup-ayrıklığı doğrudan sınanabilir. **Yeniden eğitim gerekmez.**

In [ ]:
# ============================================================
# HUCRE 6 — ASAMA 2: BOLME DOGRULAMASI
# ============================================================
with stage("2 — Bolme (split) dogrulamasi"):
    assert DRIVE_OK, "Drive/proje klasoru yok"
    import pandas as pd, numpy as np
    from sklearn.model_selection import GroupShuffleSplit

    SEED, TEST_SIZE = 42, 0.2
    split_report = {}

    def check_group(df, cols, name):
        key = (df[cols[0]].astype(str) + "__" + df[cols[1]].astype(str)
               if len(cols) >= 2 else df[cols[0]].astype(str))
        ng = key.nunique()
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
        tr, te = next(gss.split(df, groups=key))
        overlap = len(set(key.iloc[tr]) & set(key.iloc[te]))
        r = dict(strategy="group", columns=cols, n_groups=int(ng),
                 n_train=int(len(tr)), n_test=int(len(te)),
                 overlapping_groups=int(overlap), disjoint=(overlap == 0))
        print(f"  GRUP-bazli: sutunlar={cols}, {ng} benzersiz grup, "
              f"train={len(tr)}, test={len(te)}, ORTAK GRUP={overlap} "
              f"-> {'AYRIK (dogrulandi)' if overlap == 0 else 'AYRIK DEGIL (!)'}")
        return r

    def check_time(df, tcol, name):
        ts = pd.to_datetime(df[tcol], errors="coerce")
        parsed = float(ts.notna().mean())
        order = ts.sort_values(kind="mergesort").index
        cut = int(len(order) * (1 - TEST_SIZE))
        tr, te = order[:cut], order[cut:]
        disj = bool(ts.loc[tr].max() <= ts.loc[te].min())
        r = dict(strategy="temporal", column=tcol, parsed_fraction=round(parsed, 4),
                 n_train=len(tr), n_test=len(te), chronologically_disjoint=disj)
        print(f"  ZAMAN-bazli: sutun={tcol}, ayristirilan={parsed:.3f}, "
              f"train={len(tr)}, test={len(te)} -> "
              f"{'KRONOLOJIK AYRIK' if disj else 'ORTUSME VAR'}")
        return r

    print("\n[NSL-KDD]")
    print("  Resmi onceden-bolunmus KDDTrain+/KDDTest+ dosyalari kullanildi;")
    print("  turetilmis bolme yok, bolme kaynakli sizinti kanali yok.")
    split_report["NSL-KDD"] = dict(strategy="official", disjoint=True)

    # NSL-KDD resmi bolunmus dosyalari kullanir: turetilmis bolme YOKTUR,
    # dolayisiyla bu dongude YER ALMAZ (yukarida ayrica raporlandi).
    for name in ["NF-UNSW-NB15", "CICIDS2018"]:
        path = DATA.get(name)
        print(f"\n[{name}]")
        if not path or not os.path.exists(path):
            print("  [ATLANDI] Ham dosya bulunamadi.")
            split_report[name] = dict(strategy="unavailable")
            continue
        df = pd.read_csv(path, low_memory=False)
        df.columns = [c.strip() for c in df.columns]

        if name == "NF-UNSW-NB15":
            cols = [c for c in ["IPV4_SRC_ADDR", "IPV4_DST_ADDR"] if c in df.columns]
            if cols:
                split_report[name] = check_group(df, cols, name)
            else:
                print("  [UYARI] Host sutunlari YOK -> grup garantisi gecerli DEGIL.")
                split_report[name] = dict(strategy="stratified_random")
        else:
            cands = [["Flow ID"], ["Src IP", "Dst IP"],
                     ["Source IP", "Destination IP"], ["Src IP Addr", "Dst IP Addr"]]
            chosen = next((c for c in cands if all(x in df.columns for x in c)), None)
            tcol = next((c for c in ["Timestamp", "timestamp", "Flow Start Time"]
                         if c in df.columns), None)
            if chosen:
                split_report[name] = check_group(df, chosen, name)
            elif tcol:
                split_report[name] = check_time(df, tcol, name)
            else:
                print("  [UYARI] Ne grup ne zaman sutunu var -> STRATIFIED RASTGELE.")
                print("  -> Bolum 3.2'de bu korpus icin ACIKCA sinirlama olarak belirtilmeli.")
                split_report[name] = dict(strategy="stratified_random", disjoint=False)
        del df

    RESULTS["splits"] = split_report

## Hücre 7 — AŞAMA 3: NF-UNSW-NB15 sızıntı incelemesi

Belirleyici test **Adım 2**'dir: her özellik için **tek bir eşikle** ulaşılabilen en
yüksek doğruluk. Bir özellik tek başına 0.99'u aşıyorsa, sızıntı kanıtlanmış olur.

In [ ]:
# ============================================================
# HUCRE 7 — ASAMA 3: SIZINTI INCELEMESI (NF-UNSW-NB15)
# ============================================================
with stage("3 — Sizinti incelemesi (NF-UNSW-NB15)"):
    path = DATA.get("NF-UNSW-NB15")
    assert DRIVE_OK, "Proje klasoru bulunamadi (Hucre 2)"
    assert path and os.path.exists(path), "NF-UNSW-NB15 CSV dosyasi bulunamadi"
    import pandas as pd, numpy as np
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score

    SEED, TOP_K = 42, 10
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    label_col  = "Label" if "Label" in df.columns else "label"
    attack_col = "Attack" if "Attack" in df.columns else label_col
    y = df[label_col].astype(int)
    X = df.drop(columns=[c for c in ["IPV4_SRC_ADDR", "IPV4_DST_ADDR",
                                     label_col, attack_col] if c in df.columns],
                errors="ignore")
    for c in X.select_dtypes(include=["object", "string"]).columns:
        X[c] = pd.factorize(X[c].astype(str))[0]
    X = X.replace([np.inf, -np.inf], np.nan)
    keep = X.notna().all(axis=1)
    X, y = X[keep].astype(np.float32), y[keep].values
    del df
    print(f"satir={len(X)}  ozellik={X.shape[1]}  pozitif oran={y.mean():.4f}\n")

    # ---- ADIM 1: RF onem siralamasi ----
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                random_state=SEED, class_weight="balanced")
    rf.fit(X, y)
    imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print(f"ADIM 1 — RF safsizlik-tabanli onem (ilk {TOP_K})")
    for n_, v in imp.head(TOP_K).items():
        print(f"    {v:7.4f}  {n_}")
    print(f"    ilk-1 onem payi: {imp.iloc[0]:.4f} | ilk-3: {imp.head(3).sum():.4f}\n")

    # ---- ADIM 2: tek-esik ayirt edicilik (BELIRLEYICI TEST) ----
    def best_single_split(x, yy, n=256):
        qs = np.unique(np.quantile(x, np.linspace(0.001, 0.999, n)))
        best = 0.0
        for t in qs:
            p = (x > t).astype(int)
            best = max(best, accuracy_score(yy, p), accuracy_score(yy, 1 - p))
        return float(best)

    print(f"ADIM 2 — TEK esikle ulasilan en yuksek dogruluk (ilk {TOP_K} ozellik)")
    single = {}
    for n_ in imp.head(TOP_K).index:
        single[n_] = best_single_split(X[n_].values, y)
        print(f"    {single[n_]:7.4f}  {n_}" +
              ("   <-- SIZINTI" if single[n_] > 0.99 else ""))
    worst = max(single, key=single.get)
    leaking = single[worst] > 0.99
    print(f"    en yuksek tek-ozellik dogrulugu: {single[worst]:.4f} ({worst})\n")

    # ---- ADIM 3: supheli ozellik cikarilinca ----
    suspect = imp.index[0]
    rf2 = RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                 random_state=SEED, class_weight="balanced")
    Xd = X.drop(columns=[suspect]); rf2.fit(Xd, y)
    a_with, a_without = rf.score(X, y), rf2.score(Xd, y)
    print(f"ADIM 3 — '{suspect}' cikarilarak yeniden egitim")
    print(f"    ornek-ici dogruluk (ile) : {a_with:.6f}")
    print(f"    ornek-ici dogruluk (siz) : {a_without:.6f}")
    print("    NOT: bu hizli bir tanidir; makale icin ayni ozellik dusurulerek")
    print("    boru hattinin GRUP-bazli bolmesiyle tutulan-set dogrulugu raporlanmali.")

    RESULTS["leakage"] = dict(
        top_features={k: float(v) for k, v in imp.head(TOP_K).items()},
        top1_importance=float(imp.iloc[0]), top3_importance=float(imp.head(3).sum()),
        single_split=single, max_single_feature=worst,
        max_single_accuracy=single[worst], leaking=bool(leaking),
        suspect=suspect, acc_with=float(a_with), acc_without=float(a_without))

## Hücre 8 — Özet ve **yapıştırılacak LaTeX**

Bu hücre üç aşamanın sonucunu birleştirir, Drive'a bir rapor yazar ve
makaledeki `\PH{}` yuvalarının yerine geçecek LaTeX parçacıklarını üretir.

In [ ]:
# ============================================================
# HUCRE 8 — OZET + LATEX CIKTISI
# ============================================================
print("=" * 68); print("DURUM OZETI"); print("=" * 68)
for k in ["1 — S6 uygulama dogrulamasi", "2 — Bolme (split) dogrulamasi",
          "3 — Sizinti incelemesi (NF-UNSW-NB15)"]:
    print(f"  {k:42s} {STATUS.get(k, 'CALISTIRILMADI')}")

snips = []

def tex(name):
    """Veri kaynakli tanimlayicilari LaTeX icin guvenli hale getirir."""
    for a, b in [("\\", "\\textbackslash{}"), ("_", "\\_"), ("%", "\\%"),
                 ("&", "\\&"), ("#", "\\#"), ("$", "\\$")]:
        name = name.replace(a, b)
    return name

def num(n):
    """1234567 -> 1\\,234\\,567 (yalnizca SAYIYA uygulanir)."""
    return f"{int(n):,}".replace(",", "\\,")

def sci(v):
    """1.23e-07 -> $1.23\\times10^{-7}$"""
    m, e = f"{v:.2e}".split("e")
    return "$" + m + "\\times10^{" + str(int(e)) + "}$"

# ---- Bolum 3.3 ----
if "check2" in RESULTS:
    c1 = RESULTS.get("check1")
    if c1 is None:
        s1 = ("not evaluated, since the reference implementation could not be "
              "obtained in the execution environment")
    elif c1 == 0.0:
        s1 = ("exactly zero, i.e.\\ the two implementations agree bitwise in "
              "single precision")
    else:
        s1 = sci(c1)
    snips.append(("BTD_methodology.tex", "\\PH{CHECK 1 value}", s1))
    snips.append(("BTD_methodology.tex", "\\PH{CHECK 2 value}", sci(RESULTS["check2"])))

# ---- Bolum 4.1 ----
def split_sentence(name, r):
    if not r: return None
    if r.get("strategy") == "group":
        cols = " and ".join("\\texttt{" + tex(c) + "}" for c in r["columns"])
        return (f"a group-aware split on {cols} over {num(r['n_groups'])} "
                f"unique groups, with zero groups shared between the two "
                f"partitions")
    if r.get("strategy") == "temporal":
        return ("a chronological split, with the training and test intervals "
                "confirmed disjoint")
    if r.get("strategy") == "stratified_random":
        return ("a stratified random split, because neither flow identifiers "
                "nor a parsable timestamp are present in the released CSV; "
                "flow-level leakage cannot be excluded for this corpus")
    return None

if "splits" in RESULTS:
    for nm, ph in [("NF-UNSW-NB15", "\\PH{strategy and disjointness result for\nNF-UNSW-NB15}"),
                   ("CICIDS2018",  "\\PH{strategy and disjointness result for\nCSE-CIC-IDS2018}")]:
        sent = split_sentence(nm, RESULTS["splits"].get(nm))
        if sent:
            snips.append(("BTD_results.tex", ph.replace("\n", " "), sent))

# ---- Bolum 5.7 ----
if "leakage" in RESULTS:
    L = RESULTS["leakage"]
    if L["leaking"]:
        txt = (f"the single feature \\texttt{{{tex(L['max_single_feature'])}}} attains "
               f"{L['max_single_accuracy']:.4f} accuracy on its own with a single "
               f"threshold, and \\texttt{{{tex(L['suspect'])}}} alone carries "
               f"{L['top1_importance']:.3f} of the total impurity-based importance "
               f"mass; removing it changes accuracy from {L['acc_with']:.4f} to "
               f"{L['acc_without']:.4f}. The corpus therefore contains a feature "
               f"that very nearly encodes the target, and results obtained on it "
               f"should be read accordingly")
    else:
        txt = (f"no single feature exceeds {L['max_single_accuracy']:.4f} accuracy "
               f"under a single threshold, the most important feature "
               f"(\\texttt{{{tex(L['suspect'])}}}) carries {L['top1_importance']:.3f} of "
               f"the importance mass, and removing it changes accuracy from "
               f"{L['acc_with']:.4f} to {L['acc_without']:.4f}. The saturation is "
               f"therefore not attributable to a single label-encoding column")
    snips.append(("BTD_discussion.tex",
                  "\\PH{state the top-ranked features and their importance mass, and whether\nperformance survives their removal}".replace("\n", " "),
                  txt))

print("\n" + "=" * 68)
print("MAKALEYE YAPISTIRILACAK METINLER")
print("=" * 68)
if not snips:
    print("  (hicbir asama tamamlanmadi)")
for f, ph, val in snips:
    print(f"\n--- {f} ---")
    print(f"  BUL     : {ph}")
    print(f"  DEGISTIR: {val}")

if DRIVE_OK:
    rep = f"{OUT_DIR}/verification_{RUN_ID}.json"
    with open(rep, "w") as fh:
        json.dump(dict(run_id=RUN_ID, status=STATUS, results=RESULTS,
                       latex_substitutions=[dict(file=f, find=p, replace=v)
                                            for f, p, v in snips]),
                  fh, indent=2, ensure_ascii=False, default=str)
    print(f"\n[KAYDEDILDI] {rep}")

print("\nTamamlandi.")